# 0 · Setup & Data

Provision the capstone's **two dedicated Postgres databases** and seed them. Use a **Postgres
database with the `pgvector` extension** for both the relational and vector-store needs.

- **`aurora_db`** — relational tables (`customers`, `products`, `orders`, `order_items`) **and** the
  `aurora_help_center` PGVector collection.
- **`aurora_checkpoints_db`** — agent/graph checkpoints (one thread per ticket).

Run this notebook first. It is fully working as-is.


In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## 1. Provision the databases
Creates both databases (idempotent) and enables the `pgvector` extension on `aurora_db`.

In [2]:
create_project_databases()

Provisioning Aurora project databases:
  • database 'aurora_db' already exists
  • ensured pgvector extension on 'aurora_db'
  • database 'aurora_checkpoints_db' already exists
Done.


## 2. Seed the relational store
Creates the four tables in `aurora_db` and inserts sample customers, products, and orders.

In [3]:
seed_orders_db(reset=True)

db = get_orders_sqldatabase()
print("Tables:", db.get_usable_table_names())
print()
print(db.run(
    "SELECT o.order_id, c.name, c.tier, o.status, o.total "
    "FROM orders o JOIN customers c ON c.customer_id = o.customer_id "
    "ORDER BY o.order_id"
))

  • database 'aurora_db' already exists
  • ensured pgvector extension on 'aurora_db'


Tables: ['customers', 'order_items', 'orders', 'products']

[(1001, 'Ada Lovelace', 'premium', 'shipped', Decimal('129.49')), (1002, 'Sam Carter', 'free', 'delivered', Decimal('49.50')), (1003, 'Mei Ling', 'premium', 'placed', Decimal('240.00'))]


## 3. Seed the knowledge base (PGVector)
Embeds the help-center policy snippets into the `aurora_help_center` collection, then does a quick
similarity search to confirm retrieval works.

In [4]:
seed_knowledge_base(reset=True)

retriever = get_knowledge_retriever(k=2)
for i, doc in enumerate(retriever.invoke("How long do refunds take?"), start=1):
    print(f"{i}. [{doc.metadata.get('topic')}] {doc.page_content[:80]}...")

  • database 'aurora_db' already exists
  • ensured pgvector extension on 'aurora_db'


1. [refunds] Refunds are issued to the original payment method within 5-7 business days once ...
2. [returns] Returns are accepted within 30 days of delivery for unused items in original pac...


## 4. Checkpointer sanity check
Confirms the isolated `aurora_checkpoints_db` is reachable and the Postgres checkpointer initialises.

In [5]:
checkpointer = create_pg_checkpointer()
print("Checkpointer ready:", type(checkpointer).__name__)

  • database 'aurora_checkpoints_db' already exists
Checkpointer ready: PostgresSaver


## Done ✅

You now have:
- `aurora_db` with relational data + a populated PGVector collection,
- `aurora_checkpoints_db` ready for durable ticket threads.

Continue with **1 · LCEL intake & routing**.